# LeetCode 3972 — Valid Subarrays With Matching Sum Digits II bằng Python

> **Bài Premium: đề bài viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.** Tên phương thức dưới đây (`countValidSubarrays`) được suy ra từ đề (không có skeleton công khai cho bài Premium) — cần đối chiếu lại với trình soạn thảo thật.

- Link: https://leetcode.com/problems/valid-subarrays-with-matching-sum-digits-ii/
- Độ khó: **Hard**
- Chủ đề: Mảng, tổng tiền tố (prefix sum), băm / đếm theo số dư, hai con trỏ (sliding window), chữ số đầu và chữ số cuối

Skeleton (suy luận, cần đối chiếu):

```python
class Solution:
    def countValidSubarrays(self, nums: list[int], x: int) -> int:
```

## 1. Giới thiệu & mục tiêu học

Cho mảng số nguyên dương `nums` và một chữ số `x` (từ 1 đến 9). Một mảng con `nums[l..r]` gọi là **hợp lệ** nếu tổng các phần tử của nó có **chữ số đầu tiên bằng `x`** và **chữ số cuối cùng cũng bằng `x`**. Đếm số mảng con hợp lệ. Với `n` tới `10^5` và `nums[i]` tới `10^9`, ta không thể duyệt `O(n^2)` mảng con.

Hai điều kiện có bản chất khác nhau:

- **Chữ số cuối** của một hiệu tiền tố `P[r] - P[l]` chỉ phụ thuộc `P[r] mod 10` và `P[l] mod 10` → bài toán đếm theo số dư.
- **Chữ số đầu** của tổng phụ thuộc **độ lớn** của tổng → tổng nằm trong một trong các khoảng `[x·10^k, (x+1)·10^k − 1]`. Vì mọi phần tử dương nên prefix sum **tăng ngặt**, và các `l` thỏa mỗi khoảng tạo thành **một đoạn liên tục** các chỉ số.

Sau notebook này bạn sẽ:

1. Dùng tổng tiền tố để biến "tổng mảng con" thành hiệu hai số.
2. Hiểu cách tách điều kiện "chữ số đầu" thành các khoảng giá trị theo bậc `10^k`.
3. Kết hợp **hai con trỏ** (trên prefix sum tăng) và **đếm số dư theo tiền tố** để đếm `l` thỏa hai điều kiện cùng lúc.
4. Viết 4 lời giải (vét cạn, `O(n^2)`, bisect, hai con trỏ) và đối chiếu bằng test ngẫu nhiên.
5. Phân tích `O(n · K)` với `K ≈ 15` (số chữ số tối đa của tổng).

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho `nums` (số nguyên dương) và chữ số `x`. Mảng con `nums[l..r]` (liên tiếp, không rỗng) hợp lệ khi tổng `S` của nó thỏa **cả hai**:

- chữ số đầu của `S` (khi viết thập phân) bằng `x`;
- chữ số cuối của `S` bằng `x`.

Trả về số mảng con hợp lệ.

**Ví dụ 1:** `nums = [1,100,1]`, `x = 1` → `4`.
Các mảng con hợp lệ: `[1]` (tổng 1), `[1,100]` (tổng 101), `[100,1]` (tổng 101), `[1]` cuối (tổng 1). Mảng `[100]` có tổng 100 (đầu 1, cuối 0) và `[1,100,1]` có tổng 102 (đầu 1, cuối 2) không hợp lệ.

**Ví dụ 2:** `nums = [1]`, `x = 2` → `0`.

**Ràng buộc**

- `1 <= nums.length <= 10^5`
- `1 <= nums[i] <= 10^9`
- `1 <= x <= 9`

Lưu ý: tổng lớn nhất là `10^5 × 10^9 = 10^14` — vẫn nằm gọn trong số nguyên Python (và cả `int64` ở ngôn ngữ khác).

In [1]:
examples = [
    (dict(nums=[1, 100, 1], x=1), 4),
    (dict(nums=[1], x=2), 0),
]
for args, expected in examples:
    print("nums =", args["nums"], "| x =", args["x"], "| mong đợi:", expected)

nums = [1, 100, 1] | x = 1 | mong đợi: 4
nums = [1] | x = 2 | mong đợi: 0


## 3. Kiến thức nền

### 3.1. Tổng tiền tố (prefix sum)

`P[0] = 0`, `P[i] = nums[0] + ... + nums[i-1]`. Khi đó tổng mảng con `nums[l..r]` (chỉ số 0) bằng `P[r+1] - P[l]`. Nếu mọi `nums[i] > 0` thì `P` **tăng ngặt**: `P[0] < P[1] < ... < P[n]`. Tính chất này là then chốt của bài.

In [2]:
nums = [1, 100, 1]
P = [0]
for v in nums:
    P.append(P[-1] + v)
print("P =", P)                       # [0, 1, 101, 102]
l, r = 1, 2                           # mảng con nums[1..2] = [100, 1]
print("tổng nums[1..2] =", P[r + 1] - P[l])     # 101
print("P tăng ngặt?", all(P[i] < P[i + 1] for i in range(len(P) - 1)))

P = [0, 1, 101, 102]
tổng nums[1..2] = 101
P tăng ngặt? True


### 3.2. Chữ số cuối và chữ số đầu của một số

- **Chữ số cuối** của `S` là `S % 10`.
- **Chữ số đầu**: chia liên tục cho `10` cho tới khi `S < 10`; hoặc `int(str(S)[0])`.

Chữ số cuối của hiệu: `(a - b) % 10 = ((a % 10) - (b % 10)) % 10` — Python luôn trả số không âm với phép `%` dương.

In [3]:
def first_digit(s):
    while s >= 10:
        s //= 10
    return s

for s in [7, 101, 1999, 20000, 987654321]:
    print(s, "-> đầu:", first_digit(s), "| cuối:", s % 10, "| str cho kết quả:", int(str(s)[0]))
a, b = 1234, 589
print((a - b) % 10, ((a % 10) - (b % 10)) % 10)       # 5 5

7 -> đầu: 7 | cuối: 7 | str cho kết quả: 7
101 -> đầu: 1 | cuối: 1 | str cho kết quả: 1
1999 -> đầu: 1 | cuối: 9 | str cho kết quả: 1
20000 -> đầu: 2 | cuối: 0 | str cho kết quả: 2
987654321 -> đầu: 9 | cuối: 1 | str cho kết quả: 9
5 5


### 3.3. "Chữ số đầu bằng `x`" ⇔ tổng nằm trong một khoảng theo bậc `10^k`

Một số có chữ số đầu bằng `x` và đúng `k + 1` chữ số khi và chỉ khi nó nằm trong `[x · 10^k, (x + 1) · 10^k − 1]`.

Ví dụ `x = 1`: `k=0 → [1, 1]`, `k=1 → [10, 19]`, `k=2 → [100, 199]`, `k=3 → [1000, 1999]`... Các khoảng này **rời nhau** nên đếm riêng từng `k` rồi cộng lại không bị đếm trùng.

In [4]:
x = 1
for k in range(5):
    lo, hi = x * 10**k, (x + 1) * 10**k - 1
    print(f"k={k}: [{lo}, {hi}]")
# kiểm chứng: số nào thuộc khoảng thì chữ số đầu là x
assert all(first_digit(s) == x for k in range(5) for s in range(x * 10**k, min((x + 1) * 10**k, x * 10**k + 50)))

k=0: [1, 1]
k=1: [10, 19]
k=2: [100, 199]
k=3: [1000, 1999]
k=4: [10000, 19999]


### 3.4. Hai con trỏ trên dãy tăng và `bisect`

Với dãy `P` tăng, tập các chỉ số `l` có `A <= P[l] <= B` là một **đoạn liên tục** `[lo, hi)`: `lo = bisect_left(P, A)`, `hi = bisect_right(P, B)`. Khi `A` và `B` đều tăng dần theo `r`, `lo` và `hi` cũng chỉ tăng — ta có thể thay `bisect` bằng hai con trỏ chỉ đi tới (tổng cộng `O(n)`).

In [5]:
from bisect import bisect_left, bisect_right
P = [0, 3, 7, 12, 20, 21, 30]
A, B = 5, 21
lo, hi = bisect_left(P, A), bisect_right(P, B)
print("chỉ số l có 5 <= P[l] <= 21 :", list(range(lo, hi)), "->", P[lo:hi])

chỉ số l có 5 <= P[l] <= 21 : [2, 3, 4, 5] -> [7, 12, 20, 21]


### 3.5. Đếm theo số dư bằng bảng tiền tố `cnt[c][i]`

Ta cần "trong các chỉ số `l` thuộc `[lo, hi)`, có bao nhiêu `l` có `P[l] % 10 = c`?" cho nhiều `(lo, hi, c)`. Dựng `cnt[c][i]` = số chỉ số `j < i` có `P[j] % 10 = c`. Khi đó đáp án là `cnt[c][hi] - cnt[c][lo]`.

In [6]:
P = [0, 3, 7, 12, 20, 21, 30]
n1 = len(P)
cnt = [[0] * (n1 + 1) for _ in range(10)]
for i, v in enumerate(P):
    for c in range(10):
        cnt[c][i + 1] = cnt[c][i] + (1 if v % 10 == c else 0)
print("residues:", [v % 10 for v in P])
print("số chỉ số trong [1, 5) có dư 0:", cnt[0][5] - cnt[0][1])   # P[4]=20 -> 1
print("số chỉ số trong [0, 7) có dư 0:", cnt[0][7] - cnt[0][0])   # 0, 20, 30 -> 3

residues: [0, 3, 7, 2, 0, 1, 0]
số chỉ số trong [1, 5) có dư 0: 1
số chỉ số trong [0, 7) có dư 0: 3


## 4. Ý tưởng nền tảng / trực giác

Với ví dụ 1 (`nums = [1,100,1]`, `x = 1`): `P = [0, 1, 101, 102]`. Một mảng con ứng với một cặp `(l, r)` với `l < r` (chỉ số trong `P`), tổng `S = P[r] - P[l]`.

Liệt kê tay các cặp:

| `l` | `r` | `S` | đầu | cuối | hợp lệ? |
|---|---|---|---|---|---|
| 0 | 1 | 1 | 1 | 1 | ✓ |
| 0 | 2 | 101 | 1 | 1 | ✓ |
| 0 | 3 | 102 | 1 | 2 | ✗ |
| 1 | 2 | 100 | 1 | 0 | ✗ |
| 1 | 3 | 101 | 1 | 1 | ✓ |
| 2 | 3 | 1 | 1 | 1 | ✓ |

Tổng: `4` ✓. Thứ tự duyệt `O(n^2)` này chính là cách vét cạn. Trực giác để tăng tốc: với mỗi `r`, các `l` đáp ứng **chữ số cuối** là những `l` có `P[l] ≡ P[r] − x (mod 10)`; các `l` đáp ứng **chữ số đầu** thuộc ≤ 15 đoạn liên tục. Ta cần đếm số `l` thuộc **giao** của hai điều kiện: dùng bảng đếm theo số dư trên mỗi đoạn.

## 5. Cách 1 — Vét cạn: duyệt mọi mảng con, tính tổng rồi kiểm tra chuỗi

**Ý tưởng.** Hai vòng lặp `l`, `r`, cộng dồn tổng (hoặc dùng `sum` trực tiếp ở bản chậm nhất), chuyển thành chuỗi `str(S)` và so sánh `s[0]` và `s[-1]` với `str(x)`.

In [7]:
def brute_force(nums, x):
    n = len(nums)
    sx = str(x)
    cnt = 0
    for l in range(n):
        for r in range(l, n):
            s = str(sum(nums[l:r + 1]))            # O(n) mỗi lần: tổng O(n^3)
            if s[0] == sx and s[-1] == sx:
                cnt += 1
    return cnt

for args, expected in examples:
    got = brute_force(**args)
    print(got, expected)
    assert got == expected

4 4
0 0


**Độ phức tạp.** `O(n^3)` (tính lại tổng từng lần). Chỉ để đối chiếu với `n` nhỏ.

### 5.1. Cải thiện: tổng tiền tố, vẫn `O(n^2)`

Thay `sum(nums[l:r+1])` bằng `P[r+1] - P[l]` → `O(1)` mỗi mảng con. Dùng `% 10` và vòng `while` để lấy chữ số, không tạo chuỗi.

In [8]:
def quadratic(nums, x):
    n = len(nums)
    P = [0]
    for v in nums:
        P.append(P[-1] + v)
    cnt = 0
    for r in range(1, n + 1):
        for l in range(r):
            s = P[r] - P[l]
            if s % 10 != x:
                continue
            f = s
            while f >= 10:
                f //= 10
            if f == x:
                cnt += 1
    return cnt

for args, expected in examples:
    got = quadratic(**args)
    print(got, expected)
    assert got == expected

4 4
0 0


**Độ phức tạp.** `O(n^2 · log S)` hoặc `O(n^2)` nếu lọc chữ số cuối trước. Với `n = 10^5` là `5 · 10^9` cặp — quá chậm. **Nút thắt:** ta kiểm tra từng cặp `(l, r)` dù hầu hết đều không hợp lệ; cần **đếm** thay vì **liệt kê**.

## 6. Bước đột phá

**Insight 1 — chữ số cuối là chuyện của số dư.** `(P[r] − P[l]) % 10 = x` ⇔ `P[l] % 10 = (P[r] − x) % 10`. Với mỗi `r`, mục tiêu số dư là `c = (P[r] − x) % 10`.

**Insight 2 — chữ số đầu là chuyện của khoảng.** `S = P[r] − P[l]` có chữ số đầu `x` ⇔ tồn tại `k ≥ 0` sao cho `x·10^k ≤ S ≤ (x+1)·10^k − 1`. Suy ra `P[l]` thuộc `[P[r] − ((x+1)·10^k − 1), P[r] − x·10^k]`.

**Insight 3 — tính đơn điệu.** Vì `nums[i] ≥ 1` nên `P` **tăng ngặt**, và các `l` ứng với mỗi `k` tạo thành **một đoạn chỉ số liên tục** `[lo_k, hi_k)`. Bên cạnh đó, `x·10^k ≥ 1` nên `P[l] ≤ P[r] − 1` ⇒ `l < r` tự động.

**Insight 4 — các `k` rời nhau.** Các khoảng giá trị `[x·10^k, (x+1)·10^k − 1]` với `k` khác nhau không giao nhau (số có `k+1` chữ số khác số có `k'+1` chữ số), nên có thể cộng các kết quả theo `k` mà không đếm trùng.

**Insight 5 — số khoảng nhỏ.** `S ≤ 10^14` ⇒ `k ≤ 14`, tức tối đa `15` khoảng cho mỗi `r`.

**Công thức đáp án:**

`answer = Σ_r Σ_k #{ l ∈ [lo_k(r), hi_k(r)) : P[l] % 10 = c(r) }`

với `#` tính bằng bảng tiền tố số dư `cnt[c][·]`: `cnt[c][hi] − cnt[c][lo]`.

**Quy tắc "nếu… thì…":**

- Nếu `x·10^k > P[n]` (lớn hơn mọi tổng có thể) thì dừng, không cần xét các `k` lớn hơn.
- Nếu đoạn `[lo, hi)` rỗng (`lo >= hi`) thì đóng góp `0`.
- Nếu `nums` có thể chứa `0` hoặc số âm thì tính đơn điệu hỏng — xem phần Biến thể.

## 7. Các cách tiếp theo

### 7.1. Cách 3 — Đếm theo khoảng bằng `bisect`

**Giải thích từng đoạn.**

1. Dựng `P` (tăng ngặt) và `cnt[c][i]`: số chỉ số `j < i` có `P[j] % 10 = c`.
2. Với mỗi `k` từ `0` trong khi `x·10^k ≤ P[n]`: `lowS = x·10^k`, `highS = (x+1)·10^k − 1`.
3. Với mỗi `r` từ `1` tới `n`: `c = (P[r] − x) % 10`; `A = P[r] − highS`, `B = P[r] − lowS`.
4. `lo = bisect_left(P, A)`, `hi = bisect_right(P, B)`; cộng `cnt[c][hi] − cnt[c][lo]` nếu `hi > lo`.

In [9]:
from bisect import bisect_left, bisect_right

def build_prefix(nums):
    P = [0]
    for v in nums:
        P.append(P[-1] + v)
    return P

def build_cnt(P):
    n1 = len(P)
    cnt = [[0] * (n1 + 1) for _ in range(10)]
    for i, v in enumerate(P):
        r = v % 10
        for c in range(10):
            cnt[c][i + 1] = cnt[c][i]
        cnt[r][i + 1] += 1
    return cnt

def with_bisect(nums, x):
    n = len(nums)
    P = build_prefix(nums)
    cnt = build_cnt(P)
    total = P[n]
    ans = 0
    k, p10 = 0, 1
    while x * p10 <= total:
        lowS, highS = x * p10, (x + 1) * p10 - 1
        for r in range(1, n + 1):
            c = (P[r] - x) % 10
            lo = bisect_left(P, P[r] - highS)
            hi = bisect_right(P, P[r] - lowS)
            if hi > lo:
                ans += cnt[c][hi] - cnt[c][lo]
        k += 1
        p10 *= 10
    return ans

for args, expected in examples:
    got = with_bisect(**args)
    print(got, expected)
    assert got == expected

4 4
0 0


**Độ phức tạp.** Mỗi `(k, r)` tốn `O(log n)` cho hai lần `bisect`; tổng `O(n · K · log n)` với `K ≤ 15`; bộ nhớ `O(10 · n)` cho `cnt`. Với `n = 10^5`: `1.5·10^6` cặp `(k, r)` × 2 bisect — vài giây trong Python.

### 7.2. Cách 4 — Hai con trỏ cho mỗi `k` (bản chính, `O(n · K)`)

**Ý tưởng cải tiến.** Với `k` cố định, khi `r` tăng thì `P[r]` tăng nên cả hai biên `A = P[r] − highS`, `B = P[r] − lowS` đều tăng:

- `hi` = số chỉ số `l` có `P[l] ≤ B` — chỉ tăng; ta tiến `hi` khi `P[hi] ≤ P[r] − lowS`.
- `lo` = chỉ số đầu tiên có `P[l] ≥ A` — chỉ tăng; ta tiến `lo` khi `P[lo] < P[r] − highS`.

Mỗi con trỏ đi tối đa `n + 1` bước cho mỗi `k` ⇒ tổng `O(n)` cho mỗi `k`, không còn `log`.

**Chi tiết an toàn.** `hi` có thể tiến tối đa tới `r` (vì `P[r] − lowS < P[r]`), không vượt khỏi mảng. Ta chỉ dùng `cnt[c][hi] − cnt[c][lo]` khi `hi > lo`.

In [10]:
def two_pointers(nums, x):
    n = len(nums)
    P = build_prefix(nums)
    cnt = build_cnt(P)
    total = P[n]
    ans = 0
    p10 = 1
    while x * p10 <= total:
        lowS, highS = x * p10, (x + 1) * p10 - 1
        lo = hi = 0
        for r in range(1, n + 1):
            pr = P[r]
            while hi <= n and P[hi] <= pr - lowS:      # hi = số chỉ số có P[l] <= pr - lowS
                hi += 1
            while lo < hi and P[lo] < pr - highS:       # lo = chỉ số đầu có P[l] >= pr - highS
                lo += 1
            if hi > lo:
                c = (pr - x) % 10
                ans += cnt[c][hi] - cnt[c][lo]
        p10 *= 10
    return ans

for args, expected in examples:
    got = two_pointers(**args)
    print(got, expected)
    assert got == expected

4 4
0 0


**Độ phức tạp.** `O(n · K)` với `K ≤ 15` thời gian, `O(10 · n)` bộ nhớ cho `cnt`. Có thể giảm bộ nhớ bằng cách chỉ lưu đếm cho đúng 10 dư (như đang làm) — không phụ thuộc `K`.

## 8. TRACE từng bước

Ta trace `two_pointers` cho `nums = [1, 100, 1]`, `x = 1`. In với mỗi `k`: khoảng tổng `[lowS, highS]`, và mỗi `r`: `P[r]`, số dư đích `c`, đoạn `[lo, hi)` các chỉ số `l`, các `P[l]` trong đoạn, và đóng góp.

Bảng mong đợi (`P = [0, 1, 101, 102]`):

| `k` | khoảng `S` | `r` | `P[r]` | `c` | đoạn `l` | đóng góp |
|---|---|---|---|---|---|---|
| 0 | `[1, 1]` | 1 | 1 | 0 | `l ∈ [0,1)`: `P[0]=0` (dư 0) | 1 |
| 0 | `[1, 1]` | 2 | 101 | 0 | đoạn rỗng (cần `P[l] = 100`, không có) | 0 |
| 0 | `[1, 1]` | 3 | 102 | 1 | `l ∈ [2,3)`: `P[2]=101` (dư 1) | 1 |
| 1 | `[10, 19]` | 1..3 | — | — | không có tổng nào trong khoảng | 0 |
| 2 | `[100, 199]` | 2 | 101 | 0 | `l ∈ {0,1}` (P = 0, 1); dư 0 chỉ có `l = 0` | 1 |
| 2 | `[100, 199]` | 3 | 102 | 1 | `l ∈ {0,1}` (P = 0, 1); dư 1 chỉ có `l = 1` | 1 |

Tổng `4` ✓.

In [11]:
def trace_two_pointers(nums, x):
    n = len(nums)
    P = build_prefix(nums)
    cnt = build_cnt(P)
    total = P[n]
    print("P =", P, "| residues =", [v % 10 for v in P])
    ans, p10, k = 0, 1, 0
    while x * p10 <= total:
        lowS, highS = x * p10, (x + 1) * p10 - 1
        print(f"\n== k={k}: tổng S nằm trong [{lowS}, {highS}] ==")
        lo = hi = 0
        for r in range(1, n + 1):
            pr = P[r]
            while hi <= n and P[hi] <= pr - lowS:
                hi += 1
            while lo < hi and P[lo] < pr - highS:
                lo += 1
            c = (pr - x) % 10
            window = list(range(lo, hi))
            good = [l for l in window if P[l] % 10 == c]
            contrib = cnt[c][hi] - cnt[c][lo] if hi > lo else 0
            assert contrib == len(good)
            print(f"r={r}: P[r]={pr}, dư đích c={c}, l thuộc [{lo},{hi}) -> {window}, P[l]={[P[l] for l in window]}, l hợp lệ={good}, +{contrib}")
            ans += contrib
        p10 *= 10
        k += 1
    print("\ntổng =", ans)
    return ans

trace_two_pointers([1, 100, 1], 1)

P = [0, 1, 101, 102] | residues = [0, 1, 1, 2]

== k=0: tổng S nằm trong [1, 1] ==
r=1: P[r]=1, dư đích c=0, l thuộc [0,1) -> [0], P[l]=[0], l hợp lệ=[0], +1
r=2: P[r]=101, dư đích c=0, l thuộc [2,2) -> [], P[l]=[], l hợp lệ=[], +0
r=3: P[r]=102, dư đích c=1, l thuộc [2,3) -> [2], P[l]=[101], l hợp lệ=[2], +1

== k=1: tổng S nằm trong [10, 19] ==
r=1: P[r]=1, dư đích c=0, l thuộc [0,0) -> [], P[l]=[], l hợp lệ=[], +0
r=2: P[r]=101, dư đích c=0, l thuộc [2,2) -> [], P[l]=[], l hợp lệ=[], +0
r=3: P[r]=102, dư đích c=1, l thuộc [2,2) -> [], P[l]=[], l hợp lệ=[], +0

== k=2: tổng S nằm trong [100, 199] ==
r=1: P[r]=1, dư đích c=0, l thuộc [0,0) -> [], P[l]=[], l hợp lệ=[], +0
r=2: P[r]=101, dư đích c=0, l thuộc [0,2) -> [0, 1], P[l]=[0, 1], l hợp lệ=[0], +1
r=3: P[r]=102, dư đích c=1, l thuộc [0,2) -> [0, 1], P[l]=[0, 1], l hợp lệ=[1], +1

tổng = 4


4

In [12]:
# Một trace khác: nhiều chữ số hơn, x = 2
trace_two_pointers([2, 9, 2, 10, 5], 2)
print("brute_force:", brute_force([2, 9, 2, 10, 5], 2))

P = [0, 2, 11, 13, 23, 28] | residues = [0, 2, 1, 3, 3, 8]

== k=0: tổng S nằm trong [2, 2] ==
r=1: P[r]=2, dư đích c=0, l thuộc [0,1) -> [0], P[l]=[0], l hợp lệ=[0], +1
r=2: P[r]=11, dư đích c=9, l thuộc [2,2) -> [], P[l]=[], l hợp lệ=[], +0
r=3: P[r]=13, dư đích c=1, l thuộc [2,3) -> [2], P[l]=[11], l hợp lệ=[2], +1
r=4: P[r]=23, dư đích c=1, l thuộc [4,4) -> [], P[l]=[], l hợp lệ=[], +0
r=5: P[r]=28, dư đích c=6, l thuộc [5,5) -> [], P[l]=[], l hợp lệ=[], +0

== k=1: tổng S nằm trong [20, 29] ==
r=1: P[r]=2, dư đích c=0, l thuộc [0,0) -> [], P[l]=[], l hợp lệ=[], +0
r=2: P[r]=11, dư đích c=9, l thuộc [0,0) -> [], P[l]=[], l hợp lệ=[], +0
r=3: P[r]=13, dư đích c=1, l thuộc [0,0) -> [], P[l]=[], l hợp lệ=[], +0
r=4: P[r]=23, dư đích c=1, l thuộc [0,2) -> [0, 1], P[l]=[0, 2], l hợp lệ=[], +0
r=5: P[r]=28, dư đích c=6, l thuộc [0,2) -> [0, 1], P[l]=[0, 2], l hợp lệ=[], +0

tổng = 2
brute_force: 2


Trong trace thứ hai, bạn thấy các đoạn `[lo, hi)` dịch sang phải khi `r` tăng (hai con trỏ không bao giờ lùi), và với mỗi `k` ta chỉ quan tâm những `l` mà hiệu `P[r] − P[l]` rơi vào khoảng "chữ số đầu bằng `x`, đúng `k+1` chữ số". Đối chiếu `brute_force` cho cùng đáp án.

## 9. Code nộp LeetCode

Dùng hai con trỏ (cách 4). Lưu ý: chỉ số `cnt` kích thước `(n + 2)` cho tiện (`cnt[c][i]` với `i` từ `0` tới `n + 1`). Tên phương thức bài Premium là suy luận.

In [13]:
class Solution:
    def countValidSubarrays(self, nums: list[int], x: int) -> int:
        n = len(nums)
        P = [0] * (n + 1)                              # prefix sum, tăng ngặt vì nums[i] >= 1
        for i, v in enumerate(nums):
            P[i + 1] = P[i] + v
        # cnt[c][i] = số chỉ số j < i có P[j] % 10 == c
        cnt = [[0] * (n + 2) for _ in range(10)]
        for i in range(n + 1):
            r = P[i] % 10
            for c in range(10):
                cnt[c][i + 1] = cnt[c][i]
            cnt[r][i + 1] += 1
        total = P[n]
        ans = 0
        p10 = 1
        while x * p10 <= total:                         # k = 0, 1, 2, ...: tổng có k+1 chữ số, đầu là x
            lowS, highS = x * p10, (x + 1) * p10 - 1
            lo = hi = 0
            for r in range(1, n + 1):
                pr = P[r]
                while hi <= n and P[hi] <= pr - lowS:
                    hi += 1
                while lo < hi and P[lo] < pr - highS:
                    lo += 1
                if hi > lo:
                    c = (pr - x) % 10                   # P[l] % 10 phải bằng c để chữ số cuối là x
                    ans += cnt[c][hi] - cnt[c][lo]
            p10 *= 10
        return ans


sol = Solution()
for args, expected in examples:
    got = sol.countValidSubarrays(**args)
    print(got, expected)
    assert got == expected

4 4
0 0


## 10. Vì sao thuật toán đúng

**Tách điều kiện.** Mảng con `nums[l..r-1]` (chỉ số theo `P`, cặp `l < r`) hợp lệ ⇔ `S = P[r] − P[l]` có chữ số cuối `x` và chữ số đầu `x`.

- **Chữ số cuối:** `S mod 10 = x` ⇔ `P[l] mod 10 = (P[r] − x) mod 10 = c`. Đây là điều kiện chỉ phụ thuộc số dư.
- **Chữ số đầu:** `S ≥ 1` (vì `P` tăng ngặt) có đúng `k + 1` chữ số và chữ số đầu `x` ⇔ `x·10^k ≤ S ≤ (x+1)·10^k − 1` với **đúng một** `k ≥ 0` (số chữ số của `S` xác định `k`). Do đó đếm theo `k` rồi cộng là không trùng, không sót.

**Đoạn liên tục.** Với `k` cố định, tập `{ l : x·10^k ≤ P[r] − P[l] ≤ (x+1)·10^k − 1 }` = `{ l : P[r] − highS ≤ P[l] ≤ P[r] − lowS }`. Vì `P` tăng ngặt, đây là một đoạn chỉ số liên tục `[lo, hi)`; và vì `lowS ≥ 1` nên `hi ≤ r` (mọi `l` trong đoạn đều `< r`).

**Đếm giao.** Số `l` trong `[lo, hi)` có `P[l] mod 10 = c` bằng `cnt[c][hi] − cnt[c][lo]` theo định nghĩa của `cnt`. Cộng theo `r` và `k` ra đáp án.

**Hai con trỏ đúng.** Với `k` cố định, `P[r]` tăng nên `P[r] − lowS` và `P[r] − highS` tăng; do đó `hi = #{l : P[l] ≤ P[r] − lowS}` và `lo = #{l : P[l] < P[r] − highS}` (số chỉ số đứng trước đoạn) không giảm theo `r`. Ta khởi tạo `lo = hi = 0` và chỉ đẩy tiến.

**Điều kiện dừng `k`.** Nếu `x·10^k > P[n]` thì mọi tổng `S ≤ P[n] < x·10^k` nên không có mảng con nào thuộc khoảng đó, và các `k` lớn hơn cũng vậy.

## 11. Phân tích Time/Space

Đặt `n = len(nums)`, `K` = số chữ số tối đa của tổng (`K ≤ 15` với giới hạn đề).

| Cách | Time | Space |
|---|---|---|
| Vét cạn (tính lại tổng) | `O(n^3)` | `O(1)` |
| Tổng tiền tố | `O(n^2)` | `O(n)` |
| Khoảng + `bisect` | `O(n · K · log n)` | `O(10 · n)` |
| Khoảng + hai con trỏ (nộp) | `O(n · K)` | `O(10 · n)` |

Với `n = 10^5`, `K = 15`: khoảng `1.5·10^6` bước chính — chạy trong khoảng 1–3 giây trên Python.

In [14]:
import time, random
rng = random.Random(11)
nums = [rng.randint(1, 10**9) for _ in range(100000)]
t0 = time.time()
res = Solution().countValidSubarrays(nums, 7)
print("Solution (n=1e5, nums<=1e9, x=7):", res, f"- {time.time()-t0:.2f}s")
small = nums[:20000]
t0 = time.time(); a = with_bisect(small, 7); t1 = time.time(); b = Solution().countValidSubarrays(small, 7); t2 = time.time()
print(f"n=20000: bisect {t1-t0:.2f}s, hai con trỏ {t2-t1:.2f}s", a == b)
assert a == b

Solution (n=1e5, nums<=1e9, x=7): 19196414 - 0.41s
n=20000: bisect 0.14s, hai con trỏ 0.07s True


## 12. Bộ test

Chiến lược test:

1. 2 ví dụ của đề (assert).
2. Biên: `n = 1`; mọi phần tử `= x`; tổng cả mảng có nhiều chữ số; `x = 9` với tổng nhảy từ `9` sang `10`.
3. Phản ví dụ: kết quả khi chỉ kiểm tra chữ số cuối (xem phần 13).
4. Đối chiếu `brute_force` (O(n³)), `quadratic`, `with_bisect`, `two_pointers`, `Solution` trên nhiều mảng ngẫu nhiên có độ lớn khác nhau (từ 1 chữ số tới 6 chữ số) với seed cố định.

In [15]:
sol = Solution()
for args, expected in examples:
    assert sol.countValidSubarrays(**args) == expected

assert sol.countValidSubarrays([5], 5) == 1
assert sol.countValidSubarrays([5], 6) == 0
assert sol.countValidSubarrays([7, 7, 7], 7) == 3            # tổng 7,7,7,14,14,21: chỉ các mảng 1 phần tử; 14 đầu 1, 21 đầu 2
assert sol.countValidSubarrays([9, 1, 9], 9) == 2            # 9, 10, 19, 1, 10, 9 -> [9] và [9] cuối; 19 đầu 1; 10 đầu 1
assert sol.countValidSubarrays([1] * 12, 1) == brute_force([1] * 12, 1)
assert sol.countValidSubarrays([10**9] * 3, 1) == brute_force([10**9] * 3, 1)
print("test cố định OK")

test cố định OK


In [16]:
import random
rng = random.Random(31415)
cnt = 0
for _ in range(1500):
    n = rng.randint(1, 12)
    mag = rng.randint(0, 6)
    nums = [rng.randint(1, 10 ** rng.randint(0, mag)) for _ in range(n)]
    x = rng.randint(1, 9)
    expected = brute_force(nums, x)
    assert quadratic(nums, x) == expected, (nums, x)
    assert with_bisect(nums, x) == expected, (nums, x)
    assert two_pointers(nums, x) == expected, (nums, x)
    assert Solution().countValidSubarrays(nums, x) == expected, (nums, x)
    cnt += 1
print("đã đối chiếu", cnt, "mảng ngẫu nhiên nhỏ: tất cả khớp vét cạn")

đã đối chiếu 1500 mảng ngẫu nhiên nhỏ: tất cả khớp vét cạn


In [17]:
# Mảng vừa (n tới 300) và nhiều độ lớn: so quadratic với hai con trỏ / bisect
rng = random.Random(2718)
for _ in range(60):
    n = rng.randint(50, 300)
    mag = rng.randint(1, 9)
    nums = [rng.randint(1, 10 ** rng.randint(0, mag)) for _ in range(n)]
    x = rng.randint(1, 9)
    e = quadratic(nums, x)
    assert with_bisect(nums, x) == e == Solution().countValidSubarrays(nums, x)
print("quadratic == bisect == hai con trỏ trên 60 mảng vừa")

quadratic == bisect == hai con trỏ trên 60 mảng vừa


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ kiểm tra chữ số cuối (quên điều kiện chữ số đầu)

**Code sai:** đếm các cặp `(l, r)` có `(P[r] − P[l]) % 10 == x` bằng `Counter` rồi dừng.
**Vì sao sai:** sẽ đếm thừa những mảng con có chữ số cuối đúng nhưng chữ số đầu khác `x` (ví dụ tổng `12` hay `22` khi `x = 2` chỉ đúng cuối). Điều kiện chữ số đầu bị bỏ — xem kết quả chạy bên dưới.
**Code đúng:** thêm điều kiện khoảng theo `k`.

In [18]:
def wrong_last_digit_only(nums, x):
    from collections import Counter
    seen = Counter({0: 1})
    s, ans = 0, 0
    for v in nums:
        s += v
        ans += seen[(s - x) % 10]            # SAI: chỉ xét chữ số cuối
        seen[s % 10] += 1
    return ans

args = dict(nums=[2, 9, 2, 10, 5], x=2)
print("sai :", wrong_last_digit_only(**args), " đúng:", Solution().countValidSubarrays(**args), "(vét cạn:", brute_force(**args), ")")

sai : 3  đúng: 2 (vét cạn: 2 )


### Lỗi 2 — Dùng `bisect_left` cho biên trên (bỏ sót tổng đúng bằng biên)

**Code sai:** `hi = bisect_left(P, P[r] - lowS)`.
**Vì sao sai:** ta cần các `l` có `P[l] <= P[r] - lowS` (**bao gồm** dấu bằng, tức tổng đúng bằng `x·10^k`, như tổng `1` khi `x = 1`). `bisect_left` dừng **trước** các giá trị bằng nên bỏ sót chúng.
**Code đúng:** `hi = bisect_right(P, P[r] - lowS)`.

In [19]:
def wrong_bisect_left(nums, x):
    n = len(nums)
    P = build_prefix(nums); cnt = build_cnt(P)
    total = P[n]; ans = 0; p10 = 1
    while x * p10 <= total:
        lowS, highS = x * p10, (x + 1) * p10 - 1
        for r in range(1, n + 1):
            lo = bisect_left(P, P[r] - highS)
            hi = bisect_left(P, P[r] - lowS)            # SAI: phải là bisect_right
            if hi > lo:
                ans += cnt[(P[r] - x) % 10][hi] - cnt[(P[r] - x) % 10][lo]
        p10 *= 10
    return ans

print("sai :", wrong_bisect_left([1], 1), " đúng:", Solution().countValidSubarrays([1], 1))
print("sai :", wrong_bisect_left([1, 100, 1], 1), " đúng:", Solution().countValidSubarrays([1, 100, 1], 1))

sai : 0  đúng: 1
sai : 2  đúng: 4


### Lỗi 3 — Quên các tổng nhiều chữ số hơn (chỉ xét `k = 0` hoặc một vài `k`)

**Code sai:** chỉ xét tổng `S` có chữ số đầu bằng `x` ở các độ dài cố định (ví dụ `S ≤ 99`).
**Vì sao sai:** tổng tới `10^14` có tới 15 chữ số; phải lặp `k` cho tới khi `x·10^k > P[n]`.
**Code đúng:** vòng `while x * p10 <= total`.

### Lỗi 4 — Hai con trỏ lùi hoặc khởi tạo sai khi chuyển sang `k` mới

**Code sai:** khởi tạo `lo = hi = 0` **một lần** ngoài vòng `k`.
**Vì sao sai:** mỗi `k` có khoảng riêng; con trỏ đã tiến xa ở `k` trước sẽ bỏ sót `l` ở `k` sau (vì `lowS`, `highS` tăng theo `k`, các biên `P[r] − highS` lại **nhỏ** hơn).
**Code đúng:** đặt lại `lo = hi = 0` mỗi `k`.

### Lỗi 5 — Dùng `bisect` hoặc hai con trỏ khi `nums` có thể bằng 0

**Vì sao sai:** nếu có `0` thì `P` chỉ tăng không ngặt, `hi` có thể vượt `r` (đếm cả `l ≥ r` — mảng rỗng/âm). Đề này có `nums[i] ≥ 1` nên không xảy ra; nhưng khi biến thể cho phép `0` bạn cần giới hạn `hi ≤ r`.

## 14. Mẹo & nhận dạng pattern

- **"Chữ số cuối / chia hết / chia dư" của tổng mảng con** → tổng tiền tố + đếm theo số dư (Counter).
- **"Chữ số đầu / độ lớn / số chữ số"** → tách thành các khoảng giá trị theo bậc `10^k`; số khoảng chỉ `O(log S)`.
- **Mảng dương ⇒ tiền tố tăng ngặt ⇒ các `l` thỏa "khoảng giá trị" là một đoạn liên tục** ⇒ hai con trỏ / `bisect`.
- Muốn đếm số phần tử trong một đoạn chỉ số có thuộc tính rời rạc (số dư) ⇒ bảng đếm tiền tố theo thuộc tính (`cnt[c][i]`).
- Giao hai điều kiện = (đoạn liên tục) ∩ (thuộc tính số dư) — làm bằng "đoạn rồi đếm trong đoạn".
- Luôn kiểm tra giới hạn tổng để chặn số vòng `k` (ở đây ≤ 15).

## 15. Biến thể

1. **Chỉ yêu cầu chữ số cuối bằng `x`:** đếm bằng một `Counter` số dư tiền tố, `O(n)` (Bài 5).
2. **Chữ số đầu bằng chữ số cuối (không cố định `x`):** lặp `x` từ `1` tới `9` và cộng (kết quả `O(9 · n · K)`).
3. **`nums` có thể bằng 0 hoặc số âm:** `P` không đơn điệu; các `l` thỏa khoảng giá trị không còn là đoạn chỉ số liên tục. Dùng offline: sắp xếp theo giá trị `P`, Fenwick tree (BIT) theo số dư và theo chỉ số.
4. **Cơ số khác 10:** thay `10` bằng cơ số `b` ở mọi chỗ (`x·b^k`, `% b`).
5. **Điều kiện "chữ số đầu bằng chữ số cuối bằng `x`" áp dụng cho tích hoặc XOR:** cần tư duy hoàn toàn khác vì mất tính cộng.

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | tính tổng từng mảng con | `O(n^3)` | `O(1)` | đối chiếu `n` rất nhỏ |
| 1b. Tổng tiền tố | `P[r] − P[l]` rồi kiểm tra | `O(n^2)` | `O(n)` | `n ≤ 3000` |
| 2. Khoảng + `bisect` | đếm `l` theo khoảng và số dư | `O(n·K·log n)` | `O(10·n)` | dễ viết, đủ nhanh |
| 3. Khoảng + hai con trỏ | thay `bisect` bằng con trỏ đơn điệu | `O(n·K)` | `O(10·n)` | **nộp bài** |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `first_digit(98765)` và `first_digit(10)` bằng bao nhiêu?

**Bài 2 (tính tay).** `nums = [2, 9, 2]`, `x = 2`. Có bao nhiêu mảng con hợp lệ?

**Bài 3 (điền TODO).** Hoàn thiện `count_in_window(cnt, c, lo, hi)` trả về số chỉ số `l` thuộc `[lo, hi)` có `P[l] % 10 == c` dựa vào bảng `cnt`.

**Bài 4 (tự viết lại).** Viết `my_count(nums, x)` theo cách hai con trỏ (có thể dùng `bisect`).

**Bài 5 (biến thể).** Viết `count_last_digit(nums, x)`: số mảng con có **chỉ** chữ số cuối của tổng bằng `x` (không cần điều kiện chữ số đầu), trong `O(n)`.

In [20]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def count_in_window(cnt, c, lo, hi):
    # TODO: nếu hi <= lo trả 0; ngược lại trả cnt[c][hi] - cnt[c][lo]
    pass

# Bài 4 — TODO
def my_count(nums, x):
    # TODO 1: dựng P và cnt (build_prefix, build_cnt)
    # TODO 2: với mỗi k (p10 = 1, 10, 100...) trong khi x * p10 <= P[n]
    # TODO 3: với mỗi r: tìm lo, hi bằng bisect_left/right; cộng count_in_window(...)
    pass

# Bài 5 — TODO
def count_last_digit(nums, x):
    # TODO: Counter số dư các tiền tố đã thấy; với mỗi tiền tố mới cộng seen[(s - x) % 10]
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: chia cho 10 cho tới khi nhỏ hơn 10.
- Mức 2: `98765 → 9876 → 987 → 98 → 9`.
- Mức 3: `9` và `1`.

**Bài 2.**
- Mức 1: liệt kê 6 mảng con và tổng của chúng.
- Mức 2: `[2]=2 ✓`, `[9]=9`, `[2]=2 ✓`, `[2,9]=11`, `[9,2]=11`, `[2,9,2]=13`.
- Mức 3: chỉ hai mảng 1 phần tử hợp lệ → `2`.

**Bài 3.**
- Mức 1: `cnt[c][i]` đếm các chỉ số `< i`.
- Mức 2: hiệu `cnt[c][hi] − cnt[c][lo]` đếm các chỉ số trong `[lo, hi)`.
- Mức 3: nhớ trả `0` khi `hi <= lo`.

**Bài 4.**
- Mức 1: với mỗi `r`, `A = P[r] − highS`, `B = P[r] − lowS`.
- Mức 2: `lo = bisect_left(P, A)`, `hi = bisect_right(P, B)`.
- Mức 3: `c = (P[r] − x) % 10`.

**Bài 5.**
- Mức 1: `S = P[r] − P[l]` có `S % 10 == x` ⇔ `P[l] % 10 == (P[r] − x) % 10`.
- Mức 2: duyệt tiền tố theo thứ tự, tra `seen` trước khi thêm tiền tố hiện tại.
- Mức 3: `seen = {0: 1}` ban đầu cho `P[0] = 0`.

## 19. Đáp án tham khảo

In [21]:
# Bài 1
assert first_digit(98765) == 9 and first_digit(10) == 1
# Bài 2
assert brute_force([2, 9, 2], 2) == 2 == Solution().countValidSubarrays([2, 9, 2], 2)

# Bài 3
def count_in_window_ans(cnt, c, lo, hi):
    if hi <= lo:
        return 0
    return cnt[c][hi] - cnt[c][lo]

P_demo = build_prefix([2, 9, 2, 10, 5]); cnt_demo = build_cnt(P_demo)
for c in range(10):
    for lo in range(len(P_demo)):
        for hi in range(len(P_demo) + 1):
            direct = sum(1 for l in range(lo, hi) if l < len(P_demo) and P_demo[l] % 10 == c)
            assert count_in_window_ans(cnt_demo, c, lo, hi) == direct
print("Bài 1–3 OK")

Bài 1–3 OK


In [22]:
# Bài 4
def my_count_ans(nums, x):
    n = len(nums)
    P = build_prefix(nums)
    cnt = build_cnt(P)
    ans, p10 = 0, 1
    while x * p10 <= P[n]:
        lowS, highS = x * p10, (x + 1) * p10 - 1
        for r in range(1, n + 1):
            lo = bisect_left(P, P[r] - highS)
            hi = bisect_right(P, P[r] - lowS)
            ans += count_in_window_ans(cnt, (P[r] - x) % 10, lo, hi)
        p10 *= 10
    return ans

# Bài 5
def count_last_digit_ans(nums, x):
    seen = {0: 1}
    s, ans = 0, 0
    for v in nums:
        s += v
        ans += seen.get((s - x) % 10, 0)
        seen[s % 10] = seen.get(s % 10, 0) + 1
    return ans

def brute_last_digit(nums, x):
    n = len(nums)
    return sum(1 for l in range(n) for r in range(l, n) if sum(nums[l:r + 1]) % 10 == x)

rng = random.Random(404)
for _ in range(500):
    nums = [rng.randint(1, 10 ** rng.randint(0, 5)) for _ in range(rng.randint(1, 10))]
    x = rng.randint(1, 9)
    assert my_count_ans(nums, x) == brute_force(nums, x)
    assert count_last_digit_ans(nums, x) == brute_last_digit(nums, x)
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "chữ số đầu theo khoảng + chữ số cuối theo số dư":**

```python
P = [0] * (n + 1)
for i, v in enumerate(nums): P[i + 1] = P[i] + v          # tăng ngặt vì nums[i] >= 1
cnt = [[0] * (n + 2) for _ in range(10)]                    # cnt[c][i] = #j < i có P[j] % 10 == c
for i in range(n + 1):
    for c in range(10): cnt[c][i + 1] = cnt[c][i]
    cnt[P[i] % 10][i + 1] += 1
p10 = 1
while x * p10 <= P[n]:
    lowS, highS = x * p10, (x + 1) * p10 - 1
    lo = hi = 0                                             # đặt lại mỗi k
    for r in range(1, n + 1):
        while hi <= n and P[hi] <= P[r] - lowS: hi += 1
        while lo < hi and P[lo] < P[r] - highS: lo += 1
        if hi > lo: ans += cnt[(P[r] - x) % 10][hi] - cnt[(P[r] - x) % 10][lo]
    p10 *= 10
```

**Checklist trước khi submit:**

- [ ] Tách hai điều kiện: số dư cho chữ số cuối, khoảng `[x·10^k, (x+1)·10^k − 1]` cho chữ số đầu.
- [ ] `P` tăng ngặt (dựa vào `nums[i] ≥ 1`) ⇒ đoạn `l` liên tục, `hi ≤ r`.
- [ ] Đặt lại `lo = hi = 0` ở mỗi `k`.
- [ ] Dừng khi `x·10^k > P[n]`.
- [ ] `c = (P[r] − x) % 10` (không âm nhờ `%` của Python).
- [ ] Không đếm trùng giữa các `k` (các khoảng rời nhau).
- [ ] Tên phương thức bài Premium cần đối chiếu lại với trình soạn thảo thật.
- [ ] Đã đối chiếu vét cạn trên mảng nhỏ ngẫu nhiên đủ nhiều độ lớn.